# Home Credit：A 部分数据准备与基线模型
本 Notebook 使用申请主表完成质量检查、训练集探索、固定划分及逻辑回归基线。
数据来源：https://www.kaggle.com/c/home-credit-default-risk/data 。字段含义来自随数据提供的 HomeCredit_columns_description.csv。
TARGET=1 表示比赛定义的还款困难，并非任意定义的破产或完全不还款。
原始文件不修改。历史明细表暂不合并。GitHub 版清除了执行输出；本次验收结果见 results/a。本版本未针对验证集调参，尚需结合老师要求确认最终提交格式。

In [ ]:
from pathlib import Path
import json, warnings, platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sklearn, scipy, joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.exceptions import ConvergenceWarning
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'docs/a/split_protocol.json').exists()), None)
assert ROOT is not None, '请从项目根目录或 notebooks 目录启动 Notebook。'
DATA = ROOT / 'data/raw'
BASE = ROOT / 'data/processed/v0'
assert (DATA / 'application_train.csv').exists(), '请先按 README 下载原始 CSV 到 data/raw。'
BASE.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
SEED = 42
import sys
sys.path.insert(0, str(ROOT/'src'))
from data_pipeline import load_main_tables, fixed_split
from baseline import build_baseline
train, test, dictionary = load_main_tables(ROOT)
print('训练主表:', train.shape, '测试主表:', test.shape)
print('软件版本:', platform.python_version(), pd.__version__, sklearn.__version__)
print(dictionary.loc[dictionary['Row'].eq('TARGET'), ['Row','Description']].to_string(index=False))


## 1. 数据质量检查
全量数据仅用于结构与质量统计。模型比较相关的探索图使用划分后的训练子集。缺失率不是删除字段的唯一依据。

In [ ]:
quality = {
 'train_rows': len(train), 'train_columns': train.shape[1], 'test_rows': len(test), 'test_columns': test.shape[1],
 'duplicate_train_ids': int(train.SK_ID_CURR.duplicated().sum()),
 'duplicate_train_rows': int(train.duplicated().sum()),
 'duplicate_test_ids': int(test.SK_ID_CURR.duplicated().sum()),
 'train_test_id_overlap': int(len(set(train.SK_ID_CURR) & set(test.SK_ID_CURR))),
 'target_missing': int(train.TARGET.isna().sum()),
 'positive_count': int(train.TARGET.sum()), 'positive_rate': float(train.TARGET.mean()),
 'employment_positive_days': int((train.DAYS_EMPLOYED > 0).sum()),
 'employment_365243': int(train.DAYS_EMPLOYED.eq(365243).sum()),
 'max_income': float(train.AMT_INCOME_TOTAL.max()),
 'categorical_features': len(train.drop(columns=['TARGET','SK_ID_CURR']).select_dtypes(include='object').columns),
 'numeric_features': len(train.drop(columns=['TARGET','SK_ID_CURR']).select_dtypes(include='number').columns)
}
assert train.TARGET.isin([0,1]).all()
assert not quality['duplicate_train_ids'] and not quality['duplicate_test_ids']
assert not quality['train_test_id_overlap']
assert set(train.columns)-{'TARGET'} == set(test.columns)
profile = pd.DataFrame({'dtype':train.dtypes.astype(str), 'missing_count':train.isna().sum(), 'missing_rate':train.isna().mean(), 'unique_nonmissing':train.nunique(), 'test_missing_rate':test.isna().mean()}).sort_values('missing_rate',ascending=False)
profile.index.name='feature'
profile.to_csv(BASE/'field_profile.csv', encoding='utf-8-sig')
(BASE/'quality.json').write_text(json.dumps(quality,indent=2),encoding='utf-8')
print(json.dumps(quality,indent=2))
print('缺失率最高的十列：')
print(profile.head(10).to_string())
print('DAYS_EMPLOYED 正数取值：', train.loc[train.DAYS_EMPLOYED>0,'DAYS_EMPLOYED'].value_counts().to_dict())


## 2. 固定划分
按 TARGET 分层随机划分 80%/20%，随机种子 42。保存客户编号及所属集合；其他组员应复用该划分。测试主表不包含标签，不参与验证评分。

In [ ]:
fit_idx, val_idx, splits = fixed_split(ROOT, train)
eda = train.iloc[fit_idx].copy()
print(splits['split'].value_counts())


## 3. 基础探索
图中展示关联，不解释因果。收入和贷款金额按训练子集五分位分组，年龄由 -DAYS_BIRTH/365.25 计算。每组同时保留样本数，避免只看百分比。

In [ ]:
fig, axs=plt.subplots(1,2,figsize=(13,4.8))
counts=eda.TARGET.value_counts().sort_index()
axs[0].bar(['No difficulty (0)','Payment difficulty (1)'],counts,color=['#426b8a','#d08154'])
axs[0].set_title('Target distribution | training split')
axs[0].set_ylabel('Applications')
for i,v in enumerate(counts): axs[0].text(i,v+1500,f'{v:,} ({v/len(eda):.1%})',ha='center')
missing=eda.drop(columns=['TARGET','SK_ID_CURR']).isna().mean().sort_values().tail(12)
axs[1].barh(missing.index,missing.values,color='#426b8a')
axs[1].set_xlabel('Missing fraction'); axs[1].set_title('12 fields with most missing values')
fig.tight_layout(); fig.savefig(BASE/'01_overview.png'); plt.close(fig)
groups={
 'Age':pd.cut(-eda.DAYS_BIRTH/365.25,[0,30,40,50,60,100],right=False),
 'Income quintile':pd.qcut(eda.AMT_INCOME_TOTAL,5,duplicates='drop'),
 'Credit quintile':pd.qcut(eda.AMT_CREDIT,5,duplicates='drop')}
fig,axs=plt.subplots(1,3,figsize=(14,4.8)); group_tables=[]
for ax,(name,bins) in zip(axs,groups.items()):
    table=eda.groupby(bins,observed=True).TARGET.agg(['size','sum','mean'])
    table['group']=table.index.astype(str); table['variable']=name; group_tables.append(table.reset_index(drop=True))
    labels=table['group'].tolist() if name=='Age' else [f'Q{i+1}' for i in range(len(table))]
    ax.bar(labels,table['mean']*100,color='#426b8a'); ax.set_title(name); ax.set_ylabel('Payment difficulty (%)')
    ax.tick_params(axis='x',rotation=25)
    for i,r in enumerate(table.itertuples()): ax.text(i,r.mean*100+0.15,f'n={r.size:,}',ha='center',fontsize=7)
    ax.set_ylim(0,max(table['mean'])*125)
fig.tight_layout(); fig.savefig(BASE/'02_group_rates.png'); plt.close(fig)
group_rates=pd.concat(group_tables,ignore_index=True)
group_rates.to_csv(BASE/'group_rates.csv',index=False,encoding='utf-8-sig')
print(group_rates.to_string(index=False))


## 4. 预处理与模型
SK_ID_CURR 不用于训练。DAYS_EMPLOYED 的 365243 是不合理的正天数，作为缺失处理；不删除对应客户。金额变量采用 log1p 缓解偏态，不裁掉高收入记录。缺失率超过 80% 的字段仅依据训练子集删除，阈值事先固定。数值中位数填补并增加缺失指示、标准化；类别填充 Missing 并独热编码，忽略新类别。所有学习参数只在训练子集拟合。逻辑回归使用默认 L2 正则、C=1，不做过采样或类别加权，不进行验证集调参。

In [ ]:
X_fit = train.iloc[fit_idx]; X_val = train.iloc[val_idx]
y_fit = train.TARGET.iloc[fit_idx]; y_val = train.TARGET.iloc[val_idx]
model = build_baseline(SEED)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    model.fit(X_fit, y_fit)
convergence = [str(w.message) for w in caught if issubclass(w.category, ConvergenceWarning)]
assert not convergence, convergence
prob = model.predict_proba(X_val)[:, 1]
assert np.isfinite(prob).all() and ((prob >= 0) & (prob <= 1)).all()
preprocessor = model.named_steps['preprocess']
print('按训练缺失率删除:', preprocessor.dropped_)
print('编码后特征数:', len(preprocessor.get_feature_names_out()))
print('优化迭代数:', model.named_steps['classifier'].n_iter_)
joblib.dump(model, BASE/'baseline_pipeline.joblib')
(BASE/'preprocessing.json').write_text(json.dumps({
    'dropped': preprocessor.dropped_, 'numeric': preprocessor.numeric_,
    'categorical': preprocessor.categorical_, 'seed': SEED,
    'cleaning': 'Included in saved pipeline. Pass raw application rows; do not clean twice.'
}, indent=2), encoding='utf-8')


## 5. 验证结果
ROC-AUC 衡量排序区分能力。Average precision 作为类别不平衡的辅助指标。0.5 阈值只用于展示混淆矩阵，不是业务决策阈值。概率尚未校准。常数基线一律输出训练集正类比例；全预测 0 也可能获得较高准确率，所以不能仅报告 accuracy。

In [ ]:
def evaluate(name,p):
    hard=(p>=0.5).astype(int)
    return {'model':name,'roc_auc':roc_auc_score(y_val,p),'average_precision':average_precision_score(y_val,p),'accuracy_at_0.5':accuracy_score(y_val,hard),'precision_at_0.5':precision_score(y_val,hard,zero_division=0),'recall_at_0.5':recall_score(y_val,hard,zero_division=0),'f1_at_0.5':f1_score(y_val,hard,zero_division=0)}
metrics=pd.DataFrame([evaluate('Constant prevalence',np.full(len(y_val),y_fit.mean())),evaluate('Logistic regression',prob)])
metrics.to_csv(BASE/'metrics.csv',index=False)
print(metrics.to_string(index=False))
print('混淆矩阵（行=真实0/1，列=预测0/1）：\n',confusion_matrix(y_val,prob>=0.5))
predictions=pd.DataFrame({'SK_ID_CURR':train.SK_ID_CURR.iloc[val_idx].to_numpy(),'TARGET':y_val.to_numpy(),'probability':prob})
predictions.to_csv(BASE/'validation_predictions.csv',index=False)
fig,axs=plt.subplots(1,2,figsize=(11,4.5))
RocCurveDisplay.from_predictions(y_val,prob,ax=axs[0],name='Logistic regression')
axs[0].plot([0,1],[0,1],linestyle='--',color='gray'); axs[0].set_title('Validation ROC')
PrecisionRecallDisplay.from_predictions(y_val,prob,ax=axs[1],name='Logistic regression')
axs[1].axhline(y_val.mean(),linestyle='--',color='gray'); axs[1].set_title('Validation precision-recall')
fig.tight_layout(); fig.savefig(BASE/'03_validation.png'); plt.close(fig)
# 检查未见类别和测试主表是否能通过同一流程。不会提交 Kaggle。
test_prob=model.predict_proba(test)[:,1]
assert len(test_prob)==len(test) and np.isfinite(test_prob).all()
print('测试主表成功通过预处理与预测:',len(test_prob),'行；该文件无标签，未评分。')


## 6. 交接与限制
复用固定客户划分。新版模型已包含清洗，直接接收原始主表行，不要再次调用清洗函数。载入模型前将 src 加入 Python 模块路径。模型仅在 80% 训练子集拟合；生成候选 CSV，不自动提交 Kaggle，不进行全量重训。当前是单次随机留出结果，不代表跨时间部署表现；历史明细表尚未合并。

## 7. 导出 V0 测试集预测
使用以上在训练子集拟合的基线生成格式正确的候选文件；不自动提交 Kaggle。验证 AUC 仍对应原留出集。后续最终模型由模型负责人确认。

In [ ]:
import sys
sys.path.insert(0, str(ROOT/'src'))
from validate_submission import validate
submission_path = ROOT/'results/submissions/submission_v0.csv'
submission_path.parent.mkdir(parents=True, exist_ok=True)
if submission_path.exists():
    raise FileExistsError('已有 submission_v0.csv，请先另存旧版本再生成。')
submission = pd.DataFrame({'SK_ID_CURR': test.SK_ID_CURR, 'TARGET': test_prob})
submission.to_csv(submission_path, index=False)
print(validate(submission_path, DATA/'sample_submission.csv', DATA/'application_test.csv'))
